# rl17_train

> Phase 17: `trainRL17`, `trainRL`'s GRPO loop (its step pieces in `rl17_update`) with three changes: each step's boards are split across team sizes (`mixN`) with scripted liars on a share of them (`liarShare`); the reference and old log-probabilities are computed once per step and reused by `epochs` update passes with a clipped reply-level ratio (`clip`); replies can be sampled under the reply schema (`guided`), and `lrFixed` switches the per-step controller off; pilot v3's fix (`valueOnly`, `tokenClip`, a longer `maxLen`) and a per-step count of cut prompts (`cut`) and the longest prompt plus reply (`longest`); Phase 18's `world` (the board options of every rollout, and the faulty slots' class); Phase 19's options of `RL19Config` (`rl19_step`: contest credit and the penalty in the rollout world, the varied-threat mix, per-agent records carried across steps, the answer halt) and `trainRL19`, a thin wrapper. At the defaults one step equals `trainRL`'s apart from the added `cut` and `longest` (tested on the tiny model), and a `trainRL19` step at the defaults equals `trainRL17`'s. Training needs the `[train]` extra; the offline tests do not.

In [ ]:
#| default_exp rl17_train

In [ ]:
#| hide
from nbdev.showdoc import *

## The training loop

In [ ]:
#| export
import json
import time
from pathlib import Path

import numpy as np

from explRL.clue_grid import BOARD, CLUE_SETTINGS
from explRL.clue_roll import coopRows
from explRL.clue_sft import libVersions, loadModel, loraConfig
from explRL.llm_runs import LLMConfig
from explRL.local_policy import LocalPolicy, pickDevice
from explRL.rl_adv import groupAdvantages, stepSamples, trainingReturns
from explRL.rl_control import invalidShare, jumpGuard, microBatches, nanMean, nextLr, progressBar, stepLine, warmLr
from explRL.rl_torch import freeGPU, loadCheckpoint, probeKL, probeLogps, restore, saveCheckpoint, saveMemory, snapshot
from explRL.rl17_update import cutCount, pairLengths, ppoUpdate, saveTurns
from explRL.rl19_grid import RL19Config, configHash19, runFields19
from explRL.rl19_bridge import bridgeStepStats
from explRL.rl19_step import RecordBook, answerStats, check19, haltReason19, playStep, world19


def makePolicies17(cfg, model, tok, dev):
    "As rl_torch.makePolicies (one shared adapter), with constrained replies when cfg.guided."
    vllm = dev == 'cuda' and LocalPolicy(cfg.model, device=dev, engine='auto', tokenizer=tok).engine == 'vllm'
    pol = LocalPolicy(cfg.model, device=dev, maxNew=cfg.maxNew, temperature=cfg.temperature, seed=cfg.trainSeed,
                      engine='vllm' if vllm else 'hf', hf=None if vllm else model, tokenizer=tok, adapterName='default',
                      guided=cfg.guided)
    return pol, vllm


def trainRL17(cfg, settings=CLUE_SETTINGS, show=True):
    "trainRL with mixed team sizes and liars, clipped passes, constrained replies and a fixed rate (rl17_grid.RL17Config)."
    import torch
    from peft import get_peft_model
    assert cfg.advantage == 'board' and not cfg.perRole, 'Phase 17 uses board-level advantages and one shared adapter'
    out, dev = Path(cfg.out), pickDevice(cfg.device)
    out.mkdir(parents=True, exist_ok=True)
    (out / 'config.json').write_text(json.dumps(dict(runFields19(cfg), hash=configHash19(cfg), versions=libVersions()), indent=1) + '\n')
    torch.manual_seed(cfg.trainSeed)
    tok, base = loadModel(cfg.model, dev)
    model = get_peft_model(base, loraConfig(cfg.loraR, cfg.loraAlpha), adapter_name='default')
    saveMemory(model, cfg.gradCkpt)
    params = [p for n, p in model.named_parameters() if 'lora_' in n]
    for p in params:
        p.requires_grad_(True)
    rate = cfg.lrFixed if cfg.lrFixed is not None else cfg.lr
    opt = torch.optim.AdamW(params, lr=rate)
    start = loadCheckpoint(model, opt, out, configHash19(cfg), ['default'])
    if (out / 'halted.json').exists():
        print('halted:', json.loads((out / 'halted.json').read_text())['reason'])
        return str(out / 'final')
    mp = out / 'metrics.jsonl'
    history = [h for h in map(json.loads, open(mp)) if h['step'] <= start] if mp.exists() else []
    mp.write_text(''.join(json.dumps(h) + '\n' for h in history))
    lr = history[-1].get('lrNext', rate) if history else rate
    book = RecordBook(out, start) if getattr(cfg, 'records', False) else None      # Phase 19, arm 4
    pol, vllm = makePolicies17(cfg, model, tok, dev)
    lcfg = lambda N, nl, life=None: LLMConfig(task='clue', channel=cfg.channel, N=N, nLiar=nl, turns=cfg.turns, model=cfg.model,
                                              world={**{k: settings[k] for k in BOARD}, 'scheme': cfg.scheme, **dict(cfg.world),
                                                     **world19(cfg, life)})
    ctl = cfg if cfg.lrFixed is None else type(cfg)(**{**cfg.__dict__, 'klTarget': 0.0})
    bar = progressBar(cfg.steps, start, f'{cfg.scheme} RL17', show)
    kept, rolls = None, 0
    with open(mp, 'a') as log:
        for s in range(start, cfg.steps):
            t0 = time.time()

            def play():
                model.eval()
                if vllm:
                    model.save_pretrained(out / 'live')
                pol.calls = s * 100_000
                if vllm:
                    pol.setAdapter(str(out / 'live'))
                return playStep(cfg, s, lcfg, pol, book)

            rowsOf = lambda gs: [r for c, ro in gs for r in coopRows(c, ro)]
            jumped = lambda gs: kept is not None and invalidShare(rowsOf(gs)) > cfg.haltJump
            gs, back, rolls = jumpGuard(play, jumped, lambda: restore(params, opt, kept), rolls)
            lr *= 0.5 ** back
            if gs is None:
                reason = f'jump: invalid replies above {cfg.haltJump:g} after two roll-backs in a row (step {s + 1})'
                (out / 'halted.json').write_text(json.dumps(dict(step=s, reason=reason)) + '\n')
                bar.write(f'halted: {reason}')
                break
            if book is not None:
                book.commit()
            Rs, signal, samples, rng = [], 0, [], np.random.default_rng([cfg.trainSeed, s])
            for c, ro in gs:
                R = trainingReturns(ro, cfg.formatPenalty, cfg.wrongPenalty)
                adv = groupAdvantages(R.reshape(-1, cfg.group, c.N)).reshape(R.shape)
                Rs.append(R[:, c.nLiar:])
                signal += len(stepSamples(ro, adv))
                samples += stepSamples(ro, adv, cfg.turnFraction, rng)
            t1 = time.time()
            model.train()
            for g in opt.param_groups:
                g['lr'] = warmLr(lr, s, cfg.warmup)
            probe = microBatches(samples[::max(1, len(samples) // cfg.probe)][:cfg.probe], cfg)
            before = probeLogps(model, tok, probe, ['default'], False, cfg.maxLen, dev)
            snap = snapshot(params, opt) if ctl.klTarget or cfg.haltJump is not None else None
            stats = ppoUpdate(model, tok, opt, params, samples, microBatches(samples, cfg), cfg.kl, cfg.maxLen, dev,
                              cfg.epochs, cfg.clip, seed=cfg.trainSeed * 100_003 + s, valueOnly=cfg.valueOnly,
                              tokenClip=cfg.tokenClip, klClamp=cfg.klClamp)
            stepKL = probeKL(before, probeLogps(model, tok, probe, ['default'], False, cfg.maxLen, dev))
            lrNext, rejected = nextLr(lr, stepKL, ctl, warming=s + 1 < cfg.warmup, trained=bool(samples))
            if rejected:
                restore(params, opt, snap)
            kept = snap if cfg.haltJump is not None else None
            rows = rowsOf(gs)
            m = dict(step=s + 1, rolloutSeconds=round(t1 - t0, 2), updateSeconds=round(time.time() - t1, 2),
                     meanReturn=float(np.mean(np.concatenate([r.ravel() for r in Rs]))),
                     welfare=float(np.mean([r['welfare'] for r in rows])), sharePerHeld=float(np.mean([r['sharePerHeld'] for r in rows])),
                     invalid=invalidShare(rows), soloShare=nanMean([r['soloShare'] for r in rows]), signalTurns=signal,
                     samples=len(samples), cut=cutCount(tok, samples, cfg.maxLen, lens := pairLengths(tok, samples)),
                     longest=max(lens, default=0), lr=warmLr(lr, s, cfg.warmup), stepKL=stepKL,
                     rejected=rejected, lrNext=lrNext,
                     gradCkpt=bool(getattr(model, 'is_gradient_checkpointing', False)), **stats,
                     **({} if cfg.haltJump is None else dict(rolledBack=back)), **answerStats(cfg, gs), **bridgeStepStats(gs, tok))
            log.write(json.dumps(m) + '\n')
            log.flush()
            history.append(m)
            bar.write(stepLine(m, cfg.steps)) if show else None
            bar.update(1)
            lr = lrNext
            reason = haltReason19(history, cfg)
            if (s + 1) % cfg.ckptEvery == 0 or s + 1 == cfg.steps or reason:
                saveCheckpoint(model, opt, s + 1, out, configHash19(cfg), None)
                saveTurns(out, s + 1, gs)
                if book is not None:
                    book.save(s + 1)
            if reason:
                (out / 'halted.json').write_text(json.dumps(dict(step=s + 1, reason=reason)) + '\n')
                bar.write(f'halted: {reason}')
                break
    bar.close()
    model.save_pretrained(out / 'final')
    del model, base, opt, params
    freeGPU(pol)
    return str(out / 'final')


def trainRL19(cfg, settings=CLUE_SETTINGS, show=True):
    "Phase 19: trainRL17 on an RL19Config (rl19_grid), refused if it cannot run (rl19_step.check19)."
    if not isinstance(cfg, RL19Config):
        raise TypeError('trainRL19 takes an RL19Config')
    check19(cfg)
    return trainRL17(cfg, settings, show)

## Tests

In [ ]:
import numpy as np
from explRL.rl17_train import *
from explRL.rl17_grid import RL17Config

In [ ]:
#| train
# With the [train] extra, no download: at the defaults trainRL17 equals trainRL (metrics and final adapter) on the tiny model;
# a mixed step with liars and clipped passes runs; its checkpoint carries configHash17
import importlib.util
if all(importlib.util.find_spec(m) for m in ('transformers', 'peft', 'tokenizers')):   # the [train] extra; the docs build runs cells with imports and lacks it
    import json
    import tempfile
    import torch
    from dataclasses import asdict
    from explRL.clue_rl import RLConfig, trainRL
    import explRL.rl17_train as T
    from explRL.tiny_model import tinyModel
    from safetensors.torch import load_file
    _tr = T.trainingReturns
    with tempfile.TemporaryDirectory() as d:
        tiny = tinyModel(d + '/tiny')
        kw = dict(steps=2, seedsPerStep=2, group=3, turns=3, ckptEvery=1, maxNew=24, lr=1e-3, haltRun=99)
        import explRL.clue_rl as C
        for mod in (C, T):                       # the tiny model's replies are invalid: the same random returns stand in for both
            rng = np.random.default_rng(0)
            mod.trainingReturns = (lambda r: (lambda ro, *a: _tr(ro, *a) + r.normal(size=np.shape(ro.returns))))(rng)
            if mod is C:
                trainRL(RLConfig(tiny, f'{d}/a', **kw))
            else:
                trainRL17(RL17Config(tiny, f'{d}/b', **kw))
        strip = lambda p: [{k: v for k, v in json.loads(x).items() if 'Seconds' not in k and k not in ('cut', 'longest')} for x in open(p)]
        assert strip(f'{d}/a/metrics.jsonl') == strip(f'{d}/b/metrics.jsonl')
        wa, wb = load_file(f'{d}/a/final/adapter_model.safetensors'), load_file(f'{d}/b/final/adapter_model.safetensors')
        assert wa.keys() == wb.keys() and all(torch.equal(wa[k], wb[k]) for k in wa)
        T.trainingReturns = (lambda r: (lambda ro, *a: _tr(ro, *a) + r.normal(size=np.shape(ro.returns))))(np.random.default_rng(1))
        cm = RL17Config(tiny, f'{d}/m', **{**kw, 'seedsPerStep': 4}, mixN=((4, 0.5), (8, 0.5)), liarShare=0.5, epochs=2, clip=0.2, kl=0.0, lrFixed=1e-3,
                        valueOnly=True, tokenClip=True, maxLen=4096)
        trainRL17(cm)
        m = [json.loads(x) for x in open(f'{d}/m/metrics.jsonl')]
        assert [x['step'] for x in m] == [1, 2] and all(x['samples'] > 0 and np.isfinite(x['loss']) for x in m)
        assert all(x['cut'] == 0 and np.isfinite(x['klAll']) and 0 < x['longest'] <= 4096 for x in m)   # pilot v3's fix runs; nothing cut at 4096
        # Phase 18 Part B's fix: every token's k3 capped, so the logged KL (value tokens and all) can never exceed the cap
        ck = RL17Config(tiny, f'{d}/k', **{**kw, 'lr': 5e-2}, kl=0.04, lrFixed=5e-2, valueOnly=True, tokenClip=True, maxLen=4096,
                        klClamp=1e-4, haltKLRun=2)
        trainRL17(ck)
        mk = [json.loads(x) for x in open(f'{d}/k/metrics.jsonl')]
        assert all(x['kl'] <= 1e-4 + 1e-12 and x['klAll'] <= 1e-4 + 1e-12 for x in mk), mk
        # Phase 18: a step on the colony board (two holders, colluding liars) through `world`
        cw = RL17Config(tiny, f'{d}/w', **{**kw, 'steps': 1, 'seedsPerStep': 2}, mixN=((8, 1.0),), liarShare=0.5, maxLen=4096,
                        world=(('colony', True), ('r', 2), ('fault', 'collude')))
        trainRL17(cw)
        from explRL.rl17_grid import configHash17
        assert json.load(open(f'{d}/w/config.json'))['hash'] == configHash17(cw) != configHash17(RL17Config(**{**asdict(cw), 'world': ()}))
        recs = [json.loads(x) for x in open(f'{d}/w/turns/step-0001.jsonl')]
        assert recs and all(r['N'] == 8 and 'Clues shared on the board:' in r['obs'] and '(agent ' not in r['obs'] for r in recs)
        assert {r['nLiar'] for r in recs} == {0, 2}
        assert json.load(open(f'{d}/m/config.json'))['hash'] == configHash17(cm)
        T.trainingReturns = _tr
        C.trainingReturns = _tr

In [ ]:
#| train
# test_rl19_step (AC-5, flag train, tiny model, no download): one trainRL19 step at the defaults equals trainRL17's (metrics and
# final adapter); a Phase 19 step (contest credit, the threat mix, records, the penalty, the answer halt) in 16-agent squads
# (8 agents in squads of 4 on the Mac) runs, never trains a liar slot, logs the answer ratio, saves its records, resumes from them, and carries configHash19
import importlib.util
if all(importlib.util.find_spec(m) for m in ('transformers', 'peft', 'tokenizers')):   # the [train] extra; the docs build runs cells with imports and lacks it
    import json
    import tempfile
    import torch
    import explRL.rl17_train as T
    from explRL.rl19_grid import LIFETIMES, VARIED, RL19Config, configHash19
    from explRL.tiny_model import tinyModel
    from safetensors.torch import load_file
    _tr = T.trainingReturns
    noisy = lambda seed: (lambda r: (lambda ro, *a: _tr(ro, *a) + r.normal(size=np.shape(ro.returns))))(np.random.default_rng(seed))
    with tempfile.TemporaryDirectory() as d:
        tiny = tinyModel(d + '/tiny')
        kw = dict(steps=1, seedsPerStep=2, group=3, turns=3, ckptEvery=1, maxNew=24, lr=1e-3, haltRun=99)
        T.trainingReturns = noisy(0)
        trainRL17(RL17Config(tiny, f'{d}/a', **kw))
        T.trainingReturns = noisy(0)
        trainRL19(RL19Config(tiny, f'{d}/b', **kw))
        strip = lambda p: [{k: v for k, v in json.loads(x).items() if 'Seconds' not in k} for x in open(p)]
        assert strip(f'{d}/a/metrics.jsonl') == strip(f'{d}/b/metrics.jsonl')
        wa, wb = load_file(f'{d}/a/final/adapter_model.safetensors'), load_file(f'{d}/b/final/adapter_model.safetensors')
        assert wa.keys() == wb.keys() and all(torch.equal(wa[k], wb[k]) for k in wa)
        assert json.load(open(f'{d}/a/config.json'))['hash'] == json.load(open(f'{d}/b/config.json'))['hash']
        warm = tuple(((8, n, L), 30.0) for n in (0, 2, 3) for L in LIFETIMES)     # liars of 16 scaled to 8; high: every step is below a third
        c19 = RL19Config(tiny, f'{d}/c', **{**kw, 'seedsPerStep': 2, 'group': 2, 'turns': 2}, mixN=((8, 1.0),), scheme='contest',
                         contestSlots='all', threat=VARIED, records=True, sharePenalty=0.5, haltAnswers=2, maxLen=1024, microBatch=2,
                         world=(('colony', True), ('r', 2), ('squads', 4), ('warmAnswers', warm)))
        T.trainingReturns = noisy(1)
        trainRL19(c19)
        trainRL19(RL19Config(**{**c19.__dict__, 'steps': 2}))           # resumes from step 1 and its records; then the answer halt
        m = [json.loads(x) for x in open(f'{d}/c/metrics.jsonl')]
        assert [x['step'] for x in m] == [1, 2] and all(x['answerRatio'] < 1 / 3 and x['answerRef'] == 30.0 * 4 for x in m)
        assert json.load(open(f'{d}/c/halted.json'))['reason'].startswith('answers per board below a third')
        assert json.load(open(f'{d}/c/config.json'))['hash'] == configHash19(c19)
        rec = json.loads(open(f'{d}/c/records/step-0002.json').read())
        from collections import Counter
        from explRL.rl19_step import threatDraws
        seen = Counter((round(n * 8 / 16), j) for s in (0, 1) for j, (n, L) in enumerate(threatDraws(c19, s)))
        assert sorted((tuple(k), len(v['boards'])) for k, v in rec) == sorted(((8, nl, j, c), n) for (nl, j), n in seen.items() for c in range(2))
        # records follow (N, liar count, board, copy): a sequence's liar slots never change; 2 boards x 2 copies over the two steps
        recs = [json.loads(x) for x in open(f'{d}/c/turns/step-0001.jsonl')]
        assert recs and all(r['N'] == 8 and r['agent'] >= r['nLiar'] for r in recs)    # liar slots are scripted: never trained
        try:
            trainRL19(RL19Config(tiny, f'{d}/e', **kw, haltAnswers=3))
            raise AssertionError('an answer halt without warmAnswers ran')
        except ValueError:
            pass
    T.trainingReturns = _tr

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()